In [3]:
# ==========================================
# IMPORT ALL LIBRARIES
# ==========================================

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split

print("✅ All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

✅ All libraries imported successfully!
PyTorch version: 2.9.1+cpu


In [4]:
# ==========================================
# DATA EXPLORATION
# ==========================================

# Dataset path
dataset_path = r"C:\Users\DELL\OneDrive\Desktop\plantvillage dataset\color"

# Get all classes
classes = sorted(os.listdir(dataset_path))

print("=" * 60)
print("DATASET EXPLORATION")
print("=" * 60)
print(f"\n✅ Total Disease Classes: {len(classes)}\n")

# Count images per class
class_counts = {}
for cls in classes:
    cls_path = os.path.join(dataset_path, cls)
    num_images = len(os.listdir(cls_path))
    class_counts[cls] = num_images

# Show first 15 classes with counts
print("📊 Sample Classes with Image Counts:\n")
for i, (cls, count) in enumerate(list(class_counts.items())[:15], 1):
    print(f"{i:2d}. {cls:50s} → {count:4d} images")

print(f"\n📈 Total Images in Dataset: {sum(class_counts.values())}")
print("=" * 60)

DATASET EXPLORATION

✅ Total Disease Classes: 38

📊 Sample Classes with Image Counts:

 1. Apple___Apple_scab                                 →  630 images
 2. Apple___Black_rot                                  →  621 images
 3. Apple___Cedar_apple_rust                           →  275 images
 4. Apple___healthy                                    → 1645 images
 5. Blueberry___healthy                                → 1502 images
 6. Cherry_(including_sour)___Powdery_mildew           → 1052 images
 7. Cherry_(including_sour)___healthy                  →  854 images
 8. Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot →  513 images
 9. Corn_(maize)___Common_rust_                        → 1192 images
10. Corn_(maize)___Northern_Leaf_Blight                →  985 images
11. Corn_(maize)___healthy                             → 1162 images
12. Grape___Black_rot                                  → 1180 images
13. Grape___Esca_(Black_Measles)                       → 1383 images
14. Grape___Leaf

In [5]:
# ==========================================
# DATA PREPROCESSING
# ==========================================

print("=" * 60)
print("DATA PREPROCESSING")
print("=" * 60)

# Step 1: Create list of all image paths and labels
all_images = []
all_labels = []

for idx, cls in enumerate(classes):
    cls_path = os.path.join(dataset_path, cls)
    for img_name in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img_name)
        all_images.append(img_path)
        all_labels.append(idx)

print(f"\n✅ Total images collected: {len(all_images)}")
print(f"✅ Total unique labels: {len(set(all_labels))}")

# Step 2: Train-Test Split (80% train, 20% test)
train_images, test_images, train_labels, test_labels = train_test_split(
    all_images, all_labels, 
    test_size=0.2, 
    random_state=42,
    stratify=all_labels
)

print(f"\n📊 Training images: {len(train_images)}")
print(f"📊 Testing images: {len(test_images)}")
print(f"📊 Split ratio: {len(train_images)/len(all_images)*100:.1f}% train, {len(test_images)/len(all_images)*100:.1f}% test")

print("\n✅ Data split complete!")
print("=" * 60)

DATA PREPROCESSING

✅ Total images collected: 54305
✅ Total unique labels: 38

📊 Training images: 43444
📊 Testing images: 10861
📊 Split ratio: 80.0% train, 20.0% test

✅ Data split complete!


In [6]:
# ==========================================
# CUSTOM DATASET CLASS
# ==========================================

class PlantDiseaseDataset(Dataset):
    """Custom Dataset for loading plant disease images"""
    
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

print("=" * 60)
print("DATASET CLASS & TRANSFORMATIONS")
print("=" * 60)
print("\n✅ Custom Dataset class created")
print("✅ Transformations defined:")
print("   - Resize to 224x224")
print("   - Convert to Tensor")
print("   - Normalize (ImageNet stats)")
print("=" * 60)

DATASET CLASS & TRANSFORMATIONS

✅ Custom Dataset class created
✅ Transformations defined:
   - Resize to 224x224
   - Convert to Tensor
   - Normalize (ImageNet stats)


In [7]:
# ==========================================
# CREATE DATALOADERS
# ==========================================

# Create dataset objects
train_dataset = PlantDiseaseDataset(train_images, train_labels, transform=transform)
test_dataset = PlantDiseaseDataset(test_images, test_labels, transform=transform)

# Create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print("=" * 60)
print("DATALOADERS CREATED")
print("=" * 60)
print(f"\n✅ Train DataLoader: {len(train_loader)} batches")
print(f"✅ Test DataLoader: {len(test_loader)} batches")
print(f"✅ Batch size: {batch_size}")
print(f"\n📦 Total training batches: {len(train_loader)}")
print(f"📦 Total testing batches: {len(test_loader)}")
print("=" * 60)

DATALOADERS CREATED

✅ Train DataLoader: 1358 batches
✅ Test DataLoader: 340 batches
✅ Batch size: 32

📦 Total training batches: 1358
📦 Total testing batches: 340
